In [22]:
!nvidia-smi

Mon Feb 12 03:10:14 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce GTX 1650        Off | 00000000:01:00.0 Off |                  N/A |
| N/A   53C    P8               3W /  50W |    980MiB /  4096MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [23]:
import os 
import pandas as pd 
import numpy as np 
import shutil 
import sys 
import tqdm.notebook as tq 
from collections import defaultdict 
import torch 
import torch.nn as nn 
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# device = torch.device('cpu')

In [24]:
data_dir = "train_data_preprocessed.csv"
df= pd.read_csv(data_dir)

In [25]:
df.head()

,content,unnecessary,mandatory,pharma,conspiracy,political,country,rushed,ingredients,side-effect,ineffective,religious,none,corrected_text,text_without_stopwords,text_lemmatized,text_without_stopwords_freq
0,imaerell BradfatherSpeak ALEXNEWMAN JOU Newsfl...,1,0,0,0,0,0,0,0,1,0,0,0,imaerell bradfatherspeak alexnewman you newsfl...,imaerell bradfatherspeak alexnewman newsflash ...,imaerell bradfatherspeak alexnewman newsflash ...,imaerell bradfatherspeak alexnewman newsflash ...
1,theousherwood LBC I am not anti vaccine but lo...,0,0,1,0,0,0,0,0,0,0,0,0,theousherwood lac i am not anti vaccine but lo...,theousherwood lac anti vaccine looking company...,theousherwood lac anti vaccine look company pf...,theousherwood lac anti look company produce re...
2,BorisJohnson I will not be taking any vaccine ...,0,0,0,0,0,0,0,0,0,0,0,1,borisjohnson i will not be taking any vaccine ...,borisjohnson taking vaccine ever get controlof...,borisjohnson take vaccine ever get controlofdi...,borisjohnson ever controlofdiseaseact1984 update
3,LPerrins They have set this up that nothing wi...,0,0,0,0,0,0,0,0,0,1,0,0,lperrins they have set this up that nothing wi...,lperrins set nothing ever opened properly new ...,lperrins set nothing ever open properly new va...,lperrins set nothing ever open properly new va...
4,AngelaDeAngelo I believe I read it on one of P...,0,0,0,0,0,0,1,0,0,0,0,0,angeladeangelo i believe i read it on one of p...,angeladeangelo believe read one pfizers partne...,angeladeangelo believe read one pfizers partne...,angeladeangelo believe read pfizers partner si...


In [45]:
# Hyperparameters
MAX_LEN = 256
TRAIN_BATCH_SIZE = 8
VALID_BATCH_SIZE = 8
TEST_BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 2e-05
THRESHOLD = 0.5 # threshold for the sigmoid

In [26]:
df_data = df[['text_without_stopwords_freq','unnecessary','mandatory','pharma','conspiracy','political','country','rushed','ingredients','side-effect','ineffective','religious','none']]

In [27]:
df_data = df_data.rename(columns={'text_without_stopwords_freq': 'concern'})


In [28]:
df_data

,concern,unnecessary,mandatory,pharma,conspiracy,political,country,rushed,ingredients,side-effect,ineffective,religious,none
0,imaerell bradfatherspeak alexnewman newsflash ...,1,0,0,0,0,0,0,0,1,0,0,0
1,theousherwood lac anti look company produce re...,0,0,1,0,0,0,0,0,0,0,0,0
2,borisjohnson ever controlofdiseaseact1984 update,0,0,0,0,0,0,0,0,0,0,0,1
3,lperrins set nothing ever open properly new va...,0,0,0,0,0,0,0,0,0,1,0,0
4,angeladeangelo believe read pfizers partner si...,0,0,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6848,need another unknown illness manufacture accid...,0,0,0,0,0,0,0,0,1,0,0,0
6849,zagamamas wary astrazenica oxford trial 28 yea...,0,0,0,0,0,0,1,0,0,0,0,0
6850,geoohhm laugh loud mean really depend mean opp...,1,0,0,0,0,0,0,0,0,0,0,0
6851,canaditude seriously care think mandatory wrong,0,1,0,0,0,0,0,0,0,0,0,0


## Using Bert

In [29]:
from transformers import BertTokenizer, BertModel

In [30]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [31]:
test_text = "We are testing BERT tokenizer."
# generate encodings
encodings = tokenizer.encode_plus(test_text, 
                                  add_special_tokens = True,
                                  max_length = 50,
                                  truncation = True,
                                  padding = "max_length", 
                                  return_attention_mask = True, 
                                  return_tensors = "pt")
encodings

{'input_ids': tensor([[  101,  2057,  2024,  5604, 14324, 19204, 17629,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]])}

In [33]:
target_list = list(df_data.columns)
target_list

['concern',
 'unnecessary',
 'mandatory',
 'pharma',
 'conspiracy',
 'political',
 'country',
 'rushed',
 'ingredients',
 'side-effect',
 'ineffective',
 'religious',
 'none']

In [34]:
target_list = target_list[1:]

In [35]:
target_list

['unnecessary',
 'mandatory',
 'pharma',
 'conspiracy',
 'political',
 'country',
 'rushed',
 'ingredients',
 'side-effect',
 'ineffective',
 'religious',
 'none']

In [36]:
class BERTClass(torch.nn.Module):
    def __init__(self):
        super(BERTClass, self).__init__()
        self.bert_model = BertModel.from_pretrained('bert-base-uncased', return_dict=True)
        self.dropout = torch.nn.Dropout(0.3)
        self.linear = torch.nn.Linear(768, 12)

    def forward(self, input_ids, attn_mask, token_type_ids):
        output = self.bert_model(
            input_ids, 
            attention_mask=attn_mask, 
            token_type_ids=token_type_ids
        )
        output_dropout = self.dropout(output.pooler_output)
        output = self.linear(output_dropout)
        return output

model = BERTClass()
model.to(device)

BERTClass(
  (bert_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [37]:
model = BERTClass()
model.load_state_dict(torch.load(os.path.join("output2","MLTC_model_state.bin")))
model = model.to(device)

In [38]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [39]:
from transformers import AdamW
optimizer = AdamW(model.parameters(), lr = 2e-5)         

/home/aayush/.local/lib/python3.10/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [40]:
# Training of the model for one epoch
def train_model(training_loader, model, optimizer):
    losses = []
    correct_predictions = 0
    num_samples = 0
    model.train()
    loop = tq.tqdm(enumerate(training_loader), total=len(training_loader), 
                      leave=True, colour='steelblue')
    for batch_idx, data in loop:
        ids = data['input_ids'].to(device, dtype = torch.long)
        mask = data['attention_mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)
        outputs = model(ids, mask, token_type_ids)
        loss = loss_fn(outputs, targets)
        losses.append(loss.item())
        outputs = torch.sigmoid(outputs).cpu().detach().numpy().round()
        targets = targets.cpu().detach().numpy()
        correct_predictions += np.sum(outputs==targets)
        num_samples += targets.size  
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    return model, float(correct_predictions)/num_samples, np.mean(losses)

### Doing Predictions

In [42]:
test_df = pd.read_csv('test_data_preprocessed.csv')

In [43]:
test_df

,id,tweet_id,content,corrected_text,text_without_stopwords,text_lemmatized,text_without_stopwords_freq
0,0,1365819211206057993t,No thanks Wonder if it will be as good as thei...,no thanks wonder if it will be as good as thei...,thanks wonder good cancer causing baby powder ...,thanks wonder good cancer cause baby powder th...,thanks wonder good cancer cause baby powder th...
1,1,1336397084891652097t,DrEricDing This vaccine is a farce it is stora...,drericding this vaccine is a farce it is stora...,drericding vaccine farce storage requirements ...,drericding vaccine farce storage requirement e...,drericding farce storage requirement extreme e...
2,2,1333458562815844352t,StefMylesTennis disclosetv CookieFreshPimp Sen...,stefmylestennis disclosetv cookiefreshpimp sen...,stefmylestennis disclosetv cookiefreshpimp sen...,stefmylestennis disclosetv cookiefreshpimp sen...,stefmylestennis disclosetv cookiefreshpimp sen...
3,3,1327908582403280896t,DL7010 You obviously do not see it as a proble...,dl7010 you obviously do not see it as a proble...,dl7010 obviously see problem fine agree disagr...,dl7010 obviously see problem fine agree disagr...,dl7010 obviously see problem fine agree disagr...
4,4,1374806491065155596t,Potso Sego Maybe good news Just read an articl...,potso sego maybe good news just read an articl...,potso sego maybe good news read article says t...,potso sego maybe good news read article say to...,potso sego maybe good news read article top vi...
...,...,...,...,...,...,...,...
2932,2932,1297985508153294848t,The unelected non licensed health expert who w...,the unelected non licensed health expert who w...,unelected non licensed health expert sued viol...,unelected non license health expert sue violat...,unelected non license health expert sue violat...
2933,2933,1343992006486458368t,Allycon3 silenced wont2 HealthFreedomIE No I h...,allycon3 silenced wont2 healthfreedomie no i h...,allycon3 silenced wont2 healthfreedomie seen r...,allycon3 silence wont2 healthfreedomie see rea...,allycon3 silence wont2 healthfreedomie see rea...
2934,2934,1377234577698226180t,BrookeOz3 Absolutely not When the same people ...,brookeoz3 absolutely not when the same people ...,brookeoz3 absolutely people proposing stop arg...,brookeoz3 absolutely people propose stop argue...,brookeoz3 absolutely propose stop argue let fl...
2935,2935,1331188026626748422t,The extortion has already begun by the airline...,the extortion has already begun by the airline...,extortion already begun airline companies way ...,extortion already begin airline company way re...,extortion already begin airline company way re...


In [46]:
text='All I know is I are not getting the vaccine Its all'
encoded_text = tokenizer.encode_plus(
    text,
    max_length=MAX_LEN,
    add_special_tokens=True,
    return_token_type_ids=True,
    pad_to_max_length=True,
    return_attention_mask=True,
    return_tensors='pt',
)
input_ids = encoded_text['input_ids'].to(device)
attention_mask = encoded_text['attention_mask'].to(device)
token_type_ids = encoded_text['token_type_ids'].to(device)
output = model(input_ids, attention_mask, token_type_ids)
 # sigmoid, for the training sigmoid is in BCEWithLogitsLoss
output = torch.sigmoid(output).detach().cpu()
# thresholding at 0.5
output = output.flatten().round().numpy()
for idx, p in enumerate(output):
  if p==1:
    print(f"Label: {target_list[idx]}")
output

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/home/aayush/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2619: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [47]:
text_list = []
output_arrays=[]
for index, row in test_df.iterrows():
    text = row['text_without_stopwords_freq']
    if not pd.isna(text):
        encoded_text = tokenizer.encode_plus(
            text,
            max_length=MAX_LEN,
            add_special_tokens=True,
            return_token_type_ids=True,
            pad_to_max_length=True,
            return_attention_mask=True,
            return_tensors='pt',    
        )
        input_ids = encoded_text['input_ids'].to(device)
        attention_mask = encoded_text['attention_mask'].to(device)
        token_type_ids = encoded_text['token_type_ids'].to(device)
        output = model(input_ids, attention_mask, token_type_ids)
        output = torch.sigmoid(output).detach().cpu()
        output = output.flatten().round().numpy()
        # print(output)
        output_arrays.append(output)
    else:
        output_arrays.append(np.zeros(12))
        print(index)
output_df = pd.DataFrame(output_arrays, columns=target_list)
test_df = pd.concat([test_df, output_df], axis=1)



2062


In [48]:

test_df = test_df.drop(columns=['content','corrected_text','text_without_stopwords','text_lemmatized','text_without_stopwords_freq'], errors='ignore')

In [49]:
test_df = test_df.drop(columns=['id','tweet_id'], errors='ignore')

In [50]:
test_df

,unnecessary,mandatory,pharma,conspiracy,political,country,rushed,ingredients,side-effect,ineffective,religious,none
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2932,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2933,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2934,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2935,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [51]:
test_df = test_df.drop(columns=['result'], errors='ignore')

In [52]:
test_df

,unnecessary,mandatory,pharma,conspiracy,political,country,rushed,ingredients,side-effect,ineffective,religious,none
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2932,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2933,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2934,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2935,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [53]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2937 entries, 0 to 2936
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   unnecessary  2937 non-null   float64
 1   mandatory    2937 non-null   float64
 2   pharma       2937 non-null   float64
 3   conspiracy   2937 non-null   float64
 4   political    2937 non-null   float64
 5   country      2937 non-null   float64
 6   rushed       2937 non-null   float64
 7   ingredients  2937 non-null   float64
 8   side-effect  2937 non-null   float64
 9   ineffective  2937 non-null   float64
 10  religious    2937 non-null   float64
 11  none         2937 non-null   float64
dtypes: float64(12)
memory usage: 275.5 KB


In [54]:
lister = []
for i in range(2937):
    row = test_df.iloc[i]
    # print(row)
    st = "";
    for k in target_list:
        # print(row[k])
        if(row[k]!=0):
            st = st+k+" "
    if(len(st)>0): lister.append(st)
    else: lister.append('none')
    # print(f'Index: {index}, Column1: {row["Column1"]}, Column2: {row["Column2"]}, Column3: {row["Column3"]}')

In [55]:
lister

['pharma ',
 'ineffective ',
 'ineffective ',
 'none',
 'rushed side-effect ',
 'none',
 'ineffective ',
 'side-effect ',
 'side-effect ',
 'ingredients ',
 'side-effect ',
 'side-effect ',
 'side-effect ',
 'rushed ',
 'side-effect ',
 'side-effect ineffective ',
 'unnecessary ineffective ',
 'rushed ',
 'mandatory ',
 'side-effect ',
 'mandatory ',
 'unnecessary ineffective ',
 'pharma ',
 'conspiracy ingredients ',
 'none ',
 'side-effect ',
 'side-effect ',
 'unnecessary side-effect ',
 'side-effect ',
 'rushed ',
 'pharma ',
 'side-effect ',
 'none',
 'ineffective ',
 'side-effect ',
 'rushed ',
 'side-effect ',
 'ineffective ',
 'side-effect ',
 'mandatory ',
 'side-effect ',
 'side-effect ',
 'rushed ',
 'pharma ',
 'none',
 'unnecessary ',
 'political ',
 'ineffective ',
 'ineffective ',
 'ineffective ',
 'unnecessary ineffective ',
 'mandatory ',
 'pharma ',
 'side-effect ',
 'mandatory ',
 'side-effect ',
 'side-effect ',
 'pharma ',
 'ineffective ',
 'political ',
 'ineffect

In [56]:
test_df['concerns'] = lister


In [57]:
test_df

,unnecessary,mandatory,pharma,conspiracy,political,country,rushed,ingredients,side-effect,ineffective,religious,none,concerns
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,pharma
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,ineffective
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,ineffective
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,none
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,rushed side-effect
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2932,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,pharma
2933,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,mandatory
2934,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,mandatory
2935,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,pharma


In [58]:
new_final = test_df[['concerns']]

In [59]:
other = pd.read_excel('test.xlsx')

In [60]:
result_df = pd.concat([other, new_final], axis=1)


In [61]:
result_df

,id,tweet_id,content,concerns
0,0,1365819211206057993t,No thanks. Wonder if it will be as good as the...,pharma
1,1,1336397084891652097t,"@DrEricDing This vaccine is a farce, it’s stor...",ineffective
2,2,1333458562815844352t,@StefMylesTennis @disclosetv @CookieFreshPimp ...,ineffective
3,3,1327908582403280896t,@DL7010 You obviously do not see it as a probl...,none
4,4,1374806491065155596t,@Potso_Sego Maybe good news. Just read an arti...,rushed side-effect
...,...,...,...,...
2932,2932,1297985508153294848t,"The unelected, non licensed, “health expert” w...",pharma
2933,2933,1343992006486458368t,@Allycon3 @silenced_wont2 @HealthFreedomIE No ...,mandatory
2934,2934,1377234577698226180t,@BrookeOz3 Absolutely not. When the same peopl...,mandatory
2935,2935,1331188026626748422t,The extortion has already begun by the airline...,pharma


In [63]:
result_df.to_csv('Binary_Roomies.csv', index=False)